# Practice Notebook — Retirement Plan Analysis

**Based on:** *Python for Engineering and Scientific Computing*, Chapter 3 (NumPy) —
the same vectorized-array, `np.linalg.solve()`, and statistical-function toolkit
used for the runtime comparison, mesh-network, and lightning-protection examples,
applied here to **retirement planning**.

This notebook has three parts, each built from a different technique in the chapter:

| Part | Retirement question | Chapter technique |
|---|---|---|
| 1 | How much will my savings grow to by retirement? | Vectorized arrays (Section 3.1.1, Listing 3.2) |
| 2 | How do I fund known future withdrawal needs exactly? | Linear systems, `np.linalg.solve()` (Section 3.4) |
| 3 | What's the risk I run out of money after I retire? | Monte Carlo + statistics (Section 3.1.5) |

## Learning objectives
1. Replace a `for`-loop compound-interest calculation with vectorized NumPy array
   math (and see why that matters, as in Listing 3.2's runtime comparison)
2. Build a cash-flow matrix from a word problem and solve `A·x = b` for the
   right bond amounts to buy (Section 3.4.1 style)
3. Run a multi-year Monte Carlo simulation across many scenarios at once using
   `np.random.normal()`, and summarize the result with `np.mean`, `np.median`,
   `np.percentile`
4. Reason about **when this kind of model is, and is not, appropriate to use**
   for real retirement decisions

## How to use this notebook
- Every task cell contains a `# TODO` and hints. Replace `None` / `...` with
  working code.
- Run cells top to bottom — later tasks depend on variables created earlier.
- Each task has an `assert` sanity check directly below it.
- Don't peek at the cheat sheet until you've tried the task yourself.
- Section 8 ("Limitations") has no code — it is short-answer reflection. It is
  the most important section for using this responsibly.


## Setup

In [ ]:
import numpy as np
from numpy.linalg import solve

np.set_printoptions(precision=2, suppress=True)


---
## Part 1 — Accumulation phase (vectorized, no loop)

You start with **\$20,000** already saved, contribute **\$500 per month**, and
expect a **7% annual return**, for **30 years** until retirement.

Instead of writing a `for` loop that adds interest and a contribution one month
at a time (slow, as Listing 3.2 demonstrated for a similar problem), build the
whole 360-month balance history as **arrays** in a few vectorized operations:

- `months` — every month index from 0 to 360, via `np.arange()`
- `monthly_rate` — convert the annual rate to an equivalent monthly rate:
  `(1 + annual_return)**(1/12) - 1`
- `growth_factors` — `(1 + monthly_rate) ** months` (an array, one growth
  factor per month)
- `balance_from_initial` — the initial balance grown by `growth_factors`
- `balance_from_contrib` — the future value of an ordinary annuity of monthly
  contributions: `monthly_contribution * ((growth_factors - 1) / monthly_rate)`
- `balance` — the sum of the two

### Task 1
**TODO:** Build all of the arrays described above.

In [ ]:
initial_balance = 20000
monthly_contribution = 500
annual_return = 0.07
years_to_retirement = 30

monthly_rate = None      # TODO
months = None             # TODO: np.arange(0, years_to_retirement*12 + 1)

growth_factors = None            # TODO
balance_from_initial = None      # TODO
balance_from_contrib = None      # TODO
balance = None                   # TODO: sum of the two balance arrays

# --- sanity check ---
assert balance.shape == months.shape
assert balance[0] == initial_balance
print(f"Balance at retirement (month {months[-1]}): ${balance[-1]:,.2f}")


---
## Part 2 — Retirement income bond ladder (cash-flow matching)

At retirement, you want to fund your **first three years** of withdrawals with
total certainty — no market risk — by buying bonds today whose cash flows
exactly match what you'll need to withdraw. This is a real technique called
**cash-flow matching** or **bond laddering**, and it is a linear system, exactly
like the mesh-current network (Table 3.2 / Listing 3.17) and the input-output
model: **`A·x = b`**.

You'll withdraw (needs grow slightly for inflation): **\$40,000** in year 1,
**\$42,000** in year 2, **\$44,000** in year 3.

Three bonds are available, each paying an annual coupon while outstanding, plus
its face value back at maturity:

| Bond | Matures (year) | Annual coupon rate |
|---|---|---|
| A | 1 | 3.0% |
| B | 2 | 3.5% |
| C | 3 | 4.0% |

For 1 unit of face value of a bond maturing in year `m` with coupon `c`: it
pays `c` at every year `i < m`, and `c + 1` (coupon **and** face value) at
year `i == m`. It pays nothing after it has matured.

### Task 2 — Build the cash-flow matrix `A`
**TODO:** `A[i-1, j]` = the cash flow paid in year `i` (1, 2, or 3) per unit
face value of bond `j` (0, 1, or 2), using the rule above. A double loop over
years and bonds is the clearest way to build this by hand.

In [ ]:
coupons = np.array([0.03, 0.035, 0.04])
maturities = np.array([1, 2, 3])
n_bonds = len(coupons)

A = np.zeros((3, 3))
# TODO: fill in A using two nested loops over year i (1..3) and bond j (0..2):
#   if i < maturities[j]:  A[i-1, j] = coupons[j]
#   if i == maturities[j]: A[i-1, j] = coupons[j] + 1
#   (otherwise leave as 0)

# --- sanity check ---
assert A.shape == (3, 3)
assert A[2, 0] == 0  # bond A has already matured by year 3
print("Cash-flow matrix A:\n", A)


### Task 3 — Build the liability vector `b`
**TODO:** `b` is the withdrawal amount needed each year, in year order.

In [ ]:
b = None  # TODO: np.array([...]) — years 1, 2, 3 withdrawal needs

# --- sanity check ---
assert b.shape == (3,)
print("Liability vector b:", b)


### Task 4 — Solve for the face value to buy of each bond
**TODO:** Solve `A x = b` for `x`, then verify `A @ x` reproduces `b`
(same verification style as Listing 3.13/3.17/3.18).

In [ ]:
x = None  # TODO: solve(A, b)

# --- sanity check ---
assert x.shape == (3,)
check = None  # TODO: A @ x  (should closely match b)
print("Face value to buy of each bond:", np.round(x, 2))
print("Check A @ x == b:", np.round(check, 2))
print("Total dollars needed today for the ladder: $%.2f" % np.sum(x))


---
## Part 3 — Monte Carlo withdrawal sustainability

The bond ladder above only covers the *first three years* with certainty.
Beyond that, your remaining portfolio (the Part 1 balance, minus what you spent
on the ladder) stays invested in the market and keeps funding withdrawals —
this is where **sequence-of-returns risk** comes in: a string of bad early
returns can deplete a portfolio even if the *average* return looks fine.

Simulate **1,000 possible 30-year futures**. Each year, the portfolio earns a
random return (mean 6%, std dev 12% — a stock/bond blend), then a fixed
**\$40,000** is withdrawn. If the balance would go negative, the plan has
"failed" (run out of money) in that scenario.

### Task 5 — Set up the simulation
**TODO:**
1. `np.random.seed(1)` for reproducibility.
2. `bal`, an array of length `n_scenarios`, starting at `start_balance`
   (use the Part 1 result, `balance[-1]`) for every scenario.
3. `depletion_year`, an array of length `n_scenarios`, all initialized to `-1`
   (meaning "not yet ruined").

In [ ]:
np.random.seed(1)
n_scenarios = 1000
n_years = 30
start_balance = balance[-1]
annual_withdrawal = 40000
mu, sigma = 0.06, 0.12

bal = None             # TODO: np.full(n_scenarios, start_balance)
depletion_year = None  # TODO: np.full(n_scenarios, -1)

# --- sanity check ---
assert bal.shape == (n_scenarios,)
assert np.all(bal == start_balance)


### Task 6 — Run the year-by-year simulation
For each of the 30 years (a Python `for` loop over years is fine here — the
*scenarios* within each year are what's vectorized, 1,000 at once):

1. Draw one random return per scenario: `np.random.normal(mu, sigma, n_scenarios)`
2. Update every scenario's balance: `bal = bal*(1+returns) - annual_withdrawal`
3. Find scenarios that just went ruined this year (`bal <= 0` **and**
   `depletion_year == -1`) and record the current year in `depletion_year`
   for those scenarios
4. Clip the balance at 0 with `np.maximum(bal, 0)` so a ruined scenario
   doesn't go further negative in later years

**TODO:** complete the loop body.

In [ ]:
for year in range(1, n_years + 1):
    returns = None       # TODO
    bal = None            # TODO: update every scenario's balance
    newly_ruined = None   # TODO: boolean mask, see step 3 above
    depletion_year[newly_ruined] = year
    bal = None             # TODO: np.maximum(bal, 0)

# --- sanity check ---
assert bal.shape == (n_scenarios,)
assert depletion_year.shape == (n_scenarios,)
print("Simulation complete.")


### Task 7 — Summarize the results
**TODO:** Using the statistical functions from Section 3.1.5, compute:
- `prob_ruin` — the fraction of scenarios where `depletion_year != -1`
  (`np.mean()` on a boolean array gives you exactly this)
- the median, mean, 10th-percentile, and 90th-percentile ending balance
  (`np.median`, `np.mean`, `np.percentile`)
- the average depletion year **among only the scenarios that were ruined**
  (filter `depletion_year` first)

In [ ]:
prob_ruin = None  # TODO
ending_balance = bal

median_bal = None       # TODO
mean_bal = None          # TODO
p10 = None                # TODO: np.percentile(ending_balance, 10)
p90 = None                # TODO: np.percentile(ending_balance, 90)

ruined = None  # TODO: depletion_year[depletion_year != -1]

print(f"Probability of depleting savings within {n_years} years: {prob_ruin*100:.1f}%")
print(f"Median ending balance: ${median_bal:,.2f}")
print(f"Mean ending balance..: ${mean_bal:,.2f}")
print(f"10th percentile......: ${p10:,.2f}")
print(f"90th percentile......: ${p90:,.2f}")
if len(ruined):
    print(f"Average depletion year (among ruined scenarios): {np.mean(ruined):.1f}")


---
## Section 8 — Limitations (reflection, no code)

This notebook combines three models, each with its own blind spots. Before
presenting numbers like these to anyone (including yourself, for a real
decision), answer the questions below in your own words.

**When is this kind of analysis a reasonable choice?**
- Getting a rough, order-of-magnitude sense of whether a savings rate and
  time horizon are "in the right neighborhood" for a goal
- Comparing the *relative* effect of choices (save $100/month more, retire
  2 years later, spend $5,000/year less) rather than trusting an absolute
  number
- Understanding sequence-of-returns risk conceptually, even with assumed
  (not historically estimated) return distributions
- Bond cash-flow matching (Part 2) for the *specific, known, near-term*
  withdrawals it was designed for — this piece is actually low-risk math,
  not a statistical estimate, as long as the bonds don't default

**When is it NOT recommended?**
- Treating the 7% return, 6%/12% Monte Carlo assumptions, or the 24.8%
  ruin probability as house-money facts. They are **assumptions you (or this
  notebook) typed in**, not measurements of the real market's future.
- Ignoring taxes, inflation on the accumulation side, Social Security /
  pension income, healthcare costs, or required minimum distributions — none
  of which are modeled here at all.
- Assuming investment returns are independent and normally distributed year
  to year. Real markets have "fat tails" (crashes are more common and more
  extreme than a normal distribution predicts) and returns are often
  correlated across consecutive years, not independent.
- Using a single fixed withdrawal amount with no adjustment for market
  performance — real retirees and advisors often use dynamic withdrawal
  rules that reduce spending after bad years, which changes the ruin
  probability substantially.
- Making an irreversible, high-stakes decision (retiring, buying an annuity,
  cashing out a pension) based only on this notebook's output. This is a
  learning tool for the technique, not a substitute for a financial planner
  working with your actual numbers, tax situation, and goals.

1. In your own words, why does the Part 3 simulation call `np.random.normal`
   inside the loop instead of drawing all 30 years of returns once outside
   the loop? Would drawing them once instead change anything about the model
   (hint: think about what varies across scenarios vs. across years)?
2. If you ran the Monte Carlo simulation with `sigma = 0.20` instead of
   `0.12`, would you expect `prob_ruin` to go up or down, even holding the
   mean return `mu` fixed? Why?
3. Name one real-world retirement risk this notebook does **not** model at
   all, and describe in one sentence how you'd extend the code to include it.
